# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signals audited below:** staleness (`freshness_tier`, feeds FlyRank's refresh flags) and
CTR-vs-position (`ctr` vs `position_tier`, feeds FlyRank's `needs_ctr_fix` logic).

**The rule, in plain words:** A page gets one of three actions, checked in this priority order
(first match wins, so a page never gets two actions):

1. **refresh** — it sits in the `91-180` freshness tier (the tier that actually showed the
   highest decline rate below, not just "old") *and* still has real traffic (≥500 impressions/90d).
2. **fix_ctr** — its CTR is below the median CTR for pages at its own position tier (it's
   underperforming its rank, not just low-traffic) *and* it has ≥100 impressions/90d and real
   position data.
3. **quick_win** — it's in "striking distance" (position 11-20, one push from page 1) *and*
   already pulls good/excellent volume (≥3,000 impressions/90d) — the fastest win per unit effort.
4. **no_action** — none of the above.

**Reason codes (one per row, matches the action):**
`stale_91_180_and_visible`, `ctr_below_tier_median`, `high_volume_striking_distance`,
`below_all_thresholds`.


In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verification-only label (NEVER a feature — trend_direction/trend_pct are label-derived,
# per the data dictionary and notebook 02's leakage demo). Used here only to check whether a
# signal is real, never fed into the score below.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"n = {len(df):,} pages | overall declining rate = {df['is_declining_label'].mean():.3f}")

# ---------------------------------------------------------------------------
# Signal check 1 — staleness (feeds FlyRank's refresh flags)
# Claim: "the longer since a page was updated, the more likely it's declining."
# ---------------------------------------------------------------------------
t1 = (df.groupby("freshness_tier")["is_declining_label"]
        .agg(n="size", declining_rate="mean")
        .reindex(["0-30", "31-90", "91-180", "181+"]))
print("\n-- Signal 1: staleness (freshness_tier) vs declining rate --")
print(t1.round(3))
print("Verdict: MIXED — decline rate rises with staleness through 0-30 -> 31-90 -> 91-180 "
      "(0.511 -> 0.589 -> 0.611, all n > 170), which supports the assumption. But the most "
      "stale tier, 181+, drops back to 0.471 — the LOWEST of all four buckets, on n=174 (above "
      "the 50-row floor, so not just noise from a tiny cell). Two honest readings: either very "
      "neglected pages already bottomed out (nothing left to decline), or 181+ is too small/odd "
      "a slice in this data to trust past 91-180. Either way, treating '181+' as the refresh "
      "signal (my first instinct) would have been wrong — 91-180 is the tier that actually "
      "earns it, and that's what the rule below uses.")

# ---------------------------------------------------------------------------
# Signal check 2 — CTR vs position (feeds FlyRank's needs_ctr_fix logic)
# Claim: "a page underperforming the CTR its own position tier normally gets is more likely
# declining" (a real CTR problem, not just a low-traffic slot).
# ---------------------------------------------------------------------------
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
print("\nMedian CTR by position tier (impressions>=100, n per tier):")
print(pd.concat([tier_median_ctr.rename("median_ctr"),
                  visible.groupby("position_tier").size().rename("n")], axis=1))

df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
mask = (df["impressions_90d"] >= 100) & (df["avg_position"] > 0)
df["ctr_underperform"] = np.where(mask, (df["ctr"] < df["tier_median_ctr"]).astype(int), np.nan)

t2 = (df.loc[mask].groupby("ctr_underperform")["is_declining_label"]
        .agg(n="size", declining_rate="mean"))
print("\n-- Signal 2: CTR underperform-for-tier vs declining rate --")
print(t2.round(3))
print("Verdict: CONFIRMED — pages below their own position tier's median CTR decline at 0.659 "
      "vs 0.543 for pages at/above it, on n=10,307 vs n=11,699 (well past the 50-row floor). "
      "This is the stronger, cleaner signal of the two — it's what carries the most weight "
      "in the rule below.")


n = 30,000 pages | overall declining rate = 0.542

-- Signal 1: staleness (freshness_tier) vs declining rate --
                    n  declining_rate
freshness_tier                       
0-30            20480           0.511
31-90             175           0.589
91-180           9171           0.611
181+              174           0.471
Verdict: MIXED — decline rate rises with staleness through 0-30 -> 31-90 -> 91-180 (0.511 -> 0.589 -> 0.611, all n > 170), which supports the assumption. But the most stale tier, 181+, drops back to 0.471 — the LOWEST of all four buckets, on n=174 (above the 50-row floor, so not just noise from a tiny cell). Two honest readings: either very neglected pages already bottomed out (nothing left to decline), or 181+ is too small/odd a slice in this data to trust past 91-180. Either way, treating '181+' as the refresh signal (my first instinct) would have been wrong — 91-180 is the tier that actually earns it, and that's what the rule below uses.

Median CTR

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

No fitted weights — every number below is a threshold or a plain ratio, readable in one pass.
Score stays in "impressions" units on purpose (impressions_90d, or impressions_90d scaled by a
0-1 gap/position term) so it means the same thing across all three actions: how much exposure
is riding on this call.


In [2]:
# Priority order — first match wins, so every page gets exactly ONE action.
refresh_cond   = (df["freshness_tier"] == "91-180") & (df["impressions_90d"] >= 500)
ctr_fix_cond   = (df["ctr"] < df["tier_median_ctr"]) & (df["impressions_90d"] >= 100) & (df["avg_position"] > 0)
quick_win_cond = (df["impression_tier"].isin(["good", "excellent"])) & (df["position_tier"] == "striking")

df["action"] = np.select(
    [refresh_cond, ctr_fix_cond, quick_win_cond],
    ["refresh", "fix_ctr", "quick_win"],
    default="no_action",
)

df["reason_code"] = np.select(
    [refresh_cond, ctr_fix_cond, quick_win_cond],
    ["stale_91_180_and_visible", "ctr_below_tier_median", "high_volume_striking_distance"],
    default="below_all_thresholds",
)

# CTR gap as a 0-1 fraction of the tier median (guards divide-by-zero on the 'deep' tier,
# whose median CTR is 0 — those rows can never satisfy ctr_fix_cond anyway since ctr >= 0).
ctr_gap = ((df["tier_median_ctr"] - df["ctr"]) / df["tier_median_ctr"].replace(0, np.nan)).clip(lower=0).fillna(0)

df["score"] = np.select(
    [refresh_cond, ctr_fix_cond, quick_win_cond],
    [df["impressions_90d"],
     df["impressions_90d"] * ctr_gap,
     df["impressions_90d"] / df["avg_position"]],
    default=0.0,
)

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print(queue["action"].value_counts())
print(f"\nscore range: {queue['score'].min():.1f} to {queue['score'].max():.1f}")

import os
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank", "content_id", "client_id", "action", "reason_code", "score",
            "impressions_90d", "days_since_last_update", "freshness_tier",
            "ctr", "avg_position", "position_tier"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("\nwrote work/outputs/baseline_action_score.csv —", len(queue), "rows")


action
no_action    15286
fix_ctr       7442
refresh       6558
quick_win      714
Name: count, dtype: int64

score range: 0.0 to 517715.0



wrote work/outputs/baseline_action_score.csv — 30000 rows


## 3. Top-10 review

*For each of the top 10: action, reason code, and what would make it wrong.*

(Task card asks for a top-10; the skeleton header above says top-20 per the general baseline
skill — going with 10 to match this week's deliverable. Happy to extend to 20 later for the
capstone.)


In [3]:
review_cols = ["rank", "content_id", "action", "reason_code", "score", "impressions_90d",
               "avg_position", "position_tier", "ctr", "freshness_tier", "trend_direction"]
top10 = queue[review_cols].head(10)
top10


,rank,content_id,action,reason_code,score,impressions_90d,avg_position,position_tier,ctr,freshness_tier,trend_direction
0,1,content_5fe46e04994d,refresh,stale_91_180_and_visible,517715.000000,517715,4.2,page_1,0.14,91-180,down
1,2,content_2dba2b1f9536,refresh,stale_91_180_and_visible,443434.000000,443434,27.9,page_3_5,0.21,91-180,stable
2,3,content_2c2606c5d176,refresh,stale_91_180_and_visible,347399.000000,347399,4.2,page_1,0.53,91-180,down
3,4,content_cb112fce36be,refresh,stale_91_180_and_visible,309910.000000,309910,5.6,page_1,0.16,91-180,down
4,5,content_9532f197bbc8,refresh,stale_91_180_and_visible,309192.000000,309192,2.0,top_3,0.87,91-180,down
5,6,content_36ff89c8214e,refresh,stale_91_180_and_visible,295097.000000,295097,7.3,page_1,0.05,91-180,stable
6,7,content_b28d1efd668f,refresh,stale_91_180_and_visible,286608.000000,286608,26.2,page_3_5,0.06,91-180,stable
7,8,content_813e88069237,refresh,stale_91_180_and_visible,233561.000000,233561,26.2,page_3_5,0.06,91-180,down
8,9,content_8451fc6f034d,fix_ctr,ctr_below_tier_median,229173.894737,272144,2.3,top_3,0.03,0-30,up
9,10,content_c21024970297,refresh,stale_91_180_and_visible,211366.000000,211366,5.1,page_1,0.41,91-180,stable


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Top-10 hand review** (`trend_direction` shown above is for this review only — it was never an
input to the score or action):

1. **#1 `refresh`** — score 517,715 from raw `impressions_90d`. `trend_direction = down`,
   consistent with the call. Would be wrong if this page's huge impression count is a one-off
   spike (a viral day) rather than steady traffic — the rule can't tell "spike" from "sustained."
2. **#2 `refresh`** — `trend_direction = stable`, not `down`. **This is a weak pick**: the rule
   flagged it purely on staleness + volume, but it isn't actually declining right now. It would
   be wrong to say "urgent" here — "worth a periodic look" is the honest claim.
3. **#3 `refresh`** — `down`, consistent, and CTR (0.53) is unusually high for `page_1`. Wrong
   if the traffic drop is seasonal rather than content decay.
4. **#4 `refresh`** — `down`, consistent. Wrong if the update 104 days ago already addressed
   the issue and results haven't caught up in GSC's reporting lag yet.
5. **#5 `refresh`** — `top_3`, `down`. Notable: a top-3 page losing traffic despite a great
   position is a stronger decay signal than the others here — a case for prioritizing it above
   the pure-volume rows near it, which the current score doesn't do (it doesn't reward position).
6. **#6 `refresh`** — `stable`, same weak-pick pattern as #2.
7. **#7 `refresh`** — `stable`, again. **Three of the top 8 are `stable`, not `down`** — see
   the pattern below.
8. **#8 `refresh`** — `down`, consistent.
9. **#9 `fix_ctr`** — CTR 0.03 vs its tier's median, on 272k impressions — clean case, and
   `trend_direction = up`. Worth double-checking: an "up"-trending page landing this high in a
   *fix* queue is exactly the case that should make a reviewer pause before spending time on it.
10. **#10 `refresh`** — `stable`.

**The real weak pattern, not just one row:** because `score = impressions_90d` for every
`refresh` row, the top of the queue is entirely dominated by whichever handful of pages happen
to have the largest raw impression counts — heavy-tailed traffic drowns out everything else
(exactly the trap the signal-auditing skill warns about, just resurfacing here in the ranking
instead of a correlation). Only 1 of the top 10 is `fix_ctr` and 0 are `quick_win`, even though
`fix_ctr` was the *stronger, cleaner* signal in the audit above (CONFIRMED, not MIXED). A
log-scaled or per-action-normalized score would very likely surface a more mixed, more honest
top 10 — that's the first thing to fix before Week 5's model has to beat this baseline.

**Leakage self-check:**
- `action`, `reason_code`, and `score` are built only from `freshness_tier`, `impressions_90d`,
  `ctr`, `avg_position`, `position_tier`, `impression_tier` — all pre-decision, observable-today
  columns.
- `trend_direction` / `trend_pct` (the label source) and `is_declining_label` appear **only** in
  the signal-check verdicts and this review table — never in `refresh_cond`, `ctr_fix_cond`,
  `quick_win_cond`, `action`, `reason_code`, or `score`.
- No FlyRank product flags (`health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`)
  are in the starter CSV at all, so none could leak in even by accident.
- No forward/future window used — every input column is a trailing-90-day-to-date measure.


In [4]:
rule_input_cols = {"freshness_tier", "impressions_90d", "ctr", "avg_position",
                    "position_tier", "impression_tier", "tier_median_ctr"}
label_cols = {"trend_direction", "trend_pct", "is_declining_label"}
assert rule_input_cols.isdisjoint(label_cols)
print("OK: rule inputs and label-derived columns do not overlap.")

# weak-pick flag: refresh rows whose trend_direction isn't actually 'down'
weak = top10[(top10["action"] == "refresh") & (top10["trend_direction"] != "down")]
print(f"\n{len(weak)} of the top 10 are 'refresh' picks where trend_direction != 'down':")
weak[["rank", "content_id", "trend_direction"]]


OK: rule inputs and label-derived columns do not overlap.

4 of the top 10 are 'refresh' picks where trend_direction != 'down':


,rank,content_id,trend_direction
1,2,content_2dba2b1f9536,stable
5,6,content_36ff89c8214e,stable
6,7,content_b28d1efd668f,stable
9,10,content_c21024970297,stable


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.